# Exploração dos dados

Pequena exploração dos dados para ter um melhor entendimento do dataset

### Load the data and first cleaning

In [27]:
import pandas as pd
import numpy as np

#load files
df_teams = pd.read_csv("hockey_development_data/Teams.csv")
df_scoring = pd.read_csv("hockey_development_data/Scoring.csv")
df_goalies = pd.read_csv("hockey_development_data/Goalies.csv")
df_coaches = pd.read_csv("hockey_development_data/Coaches.csv")

tables = [df_teams, df_scoring, df_goalies, df_coaches]

# clean spaces
for df in tables:
    df.columns = df.columns.str.strip()

# remove black spaces
def clean_strings(df):
    df_obj = df.select_dtypes(['object'])
    df[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
    return df

# clean and filter above 1990, excluding 2004 because there were no games
for i in range(len(tables)):
    tables[i] = clean_strings(tables[i])
    tables[i] = tables[i][(tables[i]['lgID'] == 'NHL')].copy()
    tables[i]['year'] = tables[i]['year'].astype(int)

df_teams, df_scoring, df_goalies, df_coaches = tables

# print("Teams")
# display(df_teams.sort_values(by=['year', 'tmID']).tail())

# print("\nCoaches")
# display(df_coaches.sort_values(by=['year', 'tmID']).head())

# print("\nScoring")
# display(df_scoring.sort_values(by=['year', 'tmID']).head())

# print("\nGoalies")
# display(df_goalies.sort_values(by=['year', 'tmID']).head())

/var/folders/ty/4m9zd4tj4nd7bct95q84x74c0000gn/T/ipykernel_81018/3804329375.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj = df.select_dtypes(['object'])
/var/folders/ty/4m9zd4tj4nd7bct95q84x74c0000gn/T/ipykernel_81018/3804329375.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/d

### Group individual stats to use in the team

In [28]:
#convert all in numbers (0 if null)
cols_scoring = ['G', 'A', 'Pts', 'PIM']
df_scoring[cols_scoring] = df_scoring[cols_scoring].apply(pd.to_numeric).fillna(0)

#group by year and sum the statistics
teams_stats = df_scoring.groupby(['tmID', 'year']).agg(
    teams_goals = ('G', 'sum'),       
    teams_pts = ('Pts', 'sum'),       
    teams_pim = ('PIM', 'sum')        
).reset_index()

#same but to the goalies
cols_goalies = ['SA', 'GA', 'Min']
df_goalies[cols_goalies] = df_goalies[cols_goalies].apply(pd.to_numeric).fillna(0)

goalie_stats = df_goalies.groupby(['tmID', 'year']).agg(
    team_SA = ('SA', 'sum'),
    team_GA = ('GA', 'sum')
).reset_index()

#Save percentage of the team
goalie_stats['team_save_pct'] = np.where(
    goalie_stats['team_SA'] > 0, 
    1 - (goalie_stats['team_GA'] / goalie_stats['team_SA']), 
    0
)

print("Teams stats")
display(teams_stats.head())

print("\nGoalie stats")
display(goalie_stats.head())

Teams stats


,tmID,year,teams_goals,teams_pts,teams_pim
0,ANA,1993,229.0,588.0,1483.0
1,ANA,1994,125.0,336.0,715.0
2,ANA,1995,234.0,637.0,1685.0
3,ANA,1996,245.0,662.0,1688.0
4,ANA,1997,205.0,523.0,1833.0



Goalie stats


,tmID,year,team_SA,team_GA,team_save_pct
0,ANA,1993,2606.0,241.0,0.907521
1,ANA,1994,1580.0,158.0,0.900000
2,ANA,1995,2634.0,242.0,0.908125
3,ANA,1996,2682.0,227.0,0.915362
4,ANA,1997,2483.0,252.0,0.898510


### Group by team

The players are part of the team, so their info should also be added in the team info

In [29]:
df_master = pd.merge(df_teams, teams_stats, on = ['tmID', 'year'], how = 'left')
df_master = pd.merge(df_master, goalie_stats, on = ['tmID', 'year'], how = 'left')

#since some years had less games we will use per game metric and not per season
cols_team_num = ['G', 'W', 'L', 'T', 'Pts', 'GF', 'GA', 'rank']
for col in cols_team_num:
    if col in df_master.columns:
        df_master[col] = pd.to_numeric(df_master[col], errors = 'coerce').fillna(0) #if the field is null (eg) coerce transforms it into NaN and fillna to 0 so we only have numbers

#dont divide by 0
df_master['G'] = df_master['G'].replace(0, 1)

df_master['win_pct'] = df_master['W'] / df_master['G']
df_master['pts_per_game'] = df_master['Pts'] / df_master['G'] 
df_master['goal_diff_per_game'] = (df_master['GF'] - df_master['GA']) / df_master['G'] 
df_master['teams_pts_per_game'] = df_master['teams_pts'] / df_master['G'] 
df_master['teams_goals_per_game'] = df_master['teams_goals'] / df_master['G']
df_master['teams_pim_per_game'] = df_master['teams_pim'] / df_master['G'] #penalties per game

display(df_master[['year', 'tmID', 'W', 'win_pct', 'pts_per_game', 'goal_diff_per_game', 'teams_pts_per_game', 'teams_goals_per_game', 'teams_pim_per_game']].head())

,year,tmID,W,win_pct,pts_per_game,goal_diff_per_game,teams_pts_per_game,teams_goals_per_game,teams_pim_per_game
0,1917,MTL,13,0.590909,1.181818,1.409091,6.909091,5.227273,15.272727
1,1917,MTW,1,0.166667,0.333333,-3.000000,3.500000,2.833333,4.500000
2,1917,OTS,9,0.409091,0.818182,-0.545455,6.954545,4.636364,15.136364
3,1917,TOA,13,0.590909,1.181818,-0.045455,7.181818,4.909091,16.136364
4,1918,MTL,10,0.555556,1.111111,0.555556,6.888889,4.888889,14.055556


### Block the info about the Y year prediction

Since to train the data to predict the Y year champion we will need to use the info about Y-1 years available #NEEDS TO BE CHECKED

In [30]:
targets = df_master[['franchID', 'year', 'rank']].copy()

#remove rank so we cant "cheat"
features = df_master.rename(columns={'rank': 'prev_rank'}).copy()
features['target_year'] = features['year'] + 1

#2004 we didnt had games
features.loc[features['year'] == 2003, 'target_year'] = 2005

df_final = pd.merge(
    targets, 
    features, 
    left_on=['franchID', 'year'], 
    right_on=['franchID', 'target_year'], 
    suffixes=('_target', '_feature')
)

df_final = df_final.rename(columns={'year_target': 'target_season', 'year_feature': 'historical_season'})
df_final = df_final.drop(columns=['target_year'])

display(df_final[['franchID', 'target_season', 'rank', 'historical_season', 'prev_rank', 'win_pct', 'goal_diff_per_game']].head())

#print(df_final.columns.tolist())
df_final.to_csv('df_final_pronto.csv', index=False)

,franchID,target_season,rank,historical_season,prev_rank,win_pct,goal_diff_per_game
0,MTL,1918,2,1917,1,0.590909,1.409091
1,STE,1918,1,1917,3,0.409091,-0.545455
2,TOR,1918,3,1917,1,0.590909,-0.045455
3,MTL,1919,2,1918,2,0.555556,0.555556
4,STE,1919,1,1918,1,0.666667,1.000000
